
# ShotSpotter vs. RMS SCA Analysis (2024–2025)

This notebook reproduces the Detroit ShotSpotter coverage analysis using **scout car areas (SCAs)** as the unit of geography.

It does five things:
1. Loads the ShotSpotter incident data, RMS crime data, and SCA GeoJSON.
2. Cleans and standardizes the SCA identifiers across sources.
3. Filters RMS records to homicides and non-fatal shootings for 2024 and 2025.
4. Classifies SCAs as ShotSpotter-covered or not covered based on 2024–2025 alerts.
5. Produces summary tables and maps for inspection.



## Inputs


In [1]:

import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 200)



## Load data

Read the three source files. The GeoJSON contains Detroit scout car area polygons; its `Area` field is the SCA identifier.


In [ ]:

shot = pd.read_csv('data/dpd_shotspotter_incident_records.csv')
rms = pd.read_csv('data/2021_2026_rms_crime_incidents.csv')
sca_gdf = gpd.read_file('data/sca_map.geojson')

print('ShotSpotter rows:', len(shot))
print('RMS rows:', len(rms))
print('SCA polygons:', len(sca_gdf))
sca_gdf[['FID', 'Area', 'Precinct']].head()



## Clean key fields

The `scout_car_area` field is stored inconsistently across the CSVs, sometimes with whitespace. This cell standardizes it to an integer SCA code and parses the main datetime fields.


In [ ]:

shot['sca'] = pd.to_numeric(shot['scout_car_area'].astype(str).str.strip(), errors='coerce')
rms['sca'] = pd.to_numeric(rms['scout_car_area'].astype(str).str.strip(), errors='coerce')

shot = shot.dropna(subset=['sca']).copy()
rms = rms.dropna(subset=['sca']).copy()

shot['sca'] = shot['sca'].astype(int)
rms['sca'] = rms['sca'].astype(int)

shot['called_at'] = pd.to_datetime(shot['called_at'], errors='coerce')
rms['incident_occurred_at'] = pd.to_datetime(rms['incident_occurred_at'], errors='coerce')

shot['year'] = shot['called_at'].dt.year
rms['year'] = rms['incident_occurred_at'].dt.year

print('ShotSpotter SCAs:', shot['sca'].nunique())
print('RMS SCAs:', rms['sca'].nunique())
print('GeoJSON SCAs:', sca_gdf['Area'].nunique())



## Validation checks

These are basic sanity checks to confirm that the SCA identifiers mostly line up across files.


In [ ]:

geo_scas = set(sca_gdf['Area'].unique())
shot_scas = set(shot['sca'].unique())
rms_scas = set(rms['sca'].unique())

print('ShotSpotter SCAs matched to GeoJSON:', len(shot_scas & geo_scas), 'of', len(shot_scas))
print('RMS SCAs matched to GeoJSON:', len(rms_scas & geo_scas), 'of', len(rms_scas))
print('RMS-only SCAs not in GeoJSON:', sorted(rms_scas - geo_scas)[:20])



## Filter RMS to shootings/homicides in 2024–2025

This analysis keeps only:
- `HOMICIDE` rows, and
- records whose `offense_description` contains `SHOOT`.

That captures the homicide and non-fatal shooting subset used in the report.


In [ ]:

rms_shoot = rms[
    rms['year'].isin([2024, 2025]) &
    (
        (rms['offense_category'] == 'HOMICIDE') |
        (rms['offense_description'].str.contains('SHOOT', case=False, na=False))
    )
].copy()

rms_shoot['offense_category'].value_counts(dropna=False)


## Define ShotSpotter coverage at the SCA level

An SCA is treated as covered if it has at least one ShotSpotter alert in 2024 or 2025.


In [ ]:
shot_2425 = shot[shot['year'].isin([2024, 2025])].copy()
coverage_scas = set(shot_2425['sca'].unique())

rms_shoot['in_coverage'] = rms_shoot['sca'].isin(coverage_scas)

print('Covered SCAs:', len(coverage_scas))
print('RMS shooting records in covered SCAs:', rms_shoot['in_coverage'].sum())
print('RMS shooting records in uncovered SCAs:', (~rms_shoot['in_coverage']).sum())

In [ ]:
#output coverage key to file
coverage_df = pd.DataFrame(coverage_scas)
coverage_df.columns = ['sca']
coverage_df.to_csv('data/coverage_scas_24_25.csv')


## Year-over-year totals by coverage group

This reproduces the headline comparison: 2024 versus 2025 shooting/homicide counts in covered versus uncovered SCAs.


In [ ]:

yearly = (
    rms_shoot.groupby(['in_coverage', 'year'])
    .size()
    .reset_index(name='shootings')
)
yearly['group'] = yearly['in_coverage'].map({True: 'ShotSpotter SCAs', False: 'No ShotSpotter SCAs'})

yearly_pivot = yearly.pivot(index='year', columns='group', values='shootings').fillna(0).astype(int)
yearly_pivot['pct_change_ss'] = ((yearly_pivot['ShotSpotter SCAs'] - yearly_pivot.shift(1)['ShotSpotter SCAs']) /
                                 yearly_pivot.shift(1)['ShotSpotter SCAs']) * 100
yearly_pivot['pct_change_no_ss'] = ((yearly_pivot['No ShotSpotter SCAs'] - yearly_pivot.shift(1)['No ShotSpotter SCAs']) /
                                    yearly_pivot.shift(1)['No ShotSpotter SCAs']) * 100

yearly_pivot


In [ ]:

fig, ax = plt.subplots(figsize=(7, 5))

groups = ['ShotSpotter SCAs', 'No ShotSpotter SCAs']
vals_2024 = [yearly_pivot.loc[2024, 'ShotSpotter SCAs'], yearly_pivot.loc[2024, 'No ShotSpotter SCAs']]
vals_2025 = [yearly_pivot.loc[2025, 'ShotSpotter SCAs'], yearly_pivot.loc[2025, 'No ShotSpotter SCAs']]

x = np.arange(len(groups))
width = 0.35

bars1 = ax.bar(x - width/2, vals_2024, width, label='2024', color='#01696f')
bars2 = ax.bar(x + width/2, vals_2025, width, label='2025', color='#4f98a3')

for bars in [bars1, bars2]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                f'{int(bar.get_height())}', ha='center', va='bottom', fontsize=10)

ax.set_xticks(x)
ax.set_xticklabels(groups)
ax.set_ylabel('RMS shootings / homicides')
ax.set_title('2024 vs. 2025 shootings/homicides by SCA coverage group')
ax.legend()
plt.tight_layout()
plt.show()



## Build the SCA summary table

This table becomes the main analytic file. It merges:
- SCA polygon metadata (`Area`, `Precinct`),
- ShotSpotter alert counts in 2024–2025,
- RMS shooting counts in 2024 and 2025,
- total change and percentage change.


In [ ]:

sca_alerts = shot_2425.groupby('sca').size().reset_index(name='alerts_2425')
sca_rms = rms_shoot.groupby(['sca', 'year']).size().reset_index(name='shootings')

sca_rms_wide = sca_rms.pivot(index='sca', columns='year', values='shootings').fillna(0).astype(int)
sca_rms_wide.columns = ['rms_2024', 'rms_2025']
sca_rms_wide['rms_total'] = sca_rms_wide['rms_2024'] + sca_rms_wide['rms_2025']
sca_rms_wide['rms_change'] = sca_rms_wide['rms_2025'] - sca_rms_wide['rms_2024']
sca_rms_wide['rms_pct_change'] = (sca_rms_wide['rms_change'] /
                                  sca_rms_wide['rms_2024'].replace(0, np.nan)) * 100
sca_rms_wide = sca_rms_wide.reset_index()

sca_table = (
    sca_gdf[['Area', 'Precinct', 'geometry']]
    .rename(columns={'Area': 'sca'})
    .merge(sca_alerts, on='sca', how='left')
    .merge(sca_rms_wide, on='sca', how='left')
)

for col in ['alerts_2425', 'rms_2024', 'rms_2025', 'rms_total', 'rms_change']:
    sca_table[col] = sca_table[col].fillna(0).astype(int)

sca_table['in_coverage'] = sca_table['alerts_2425'] > 0
sca_table['rate_per_100_alerts'] = np.where(
    sca_table['alerts_2425'] > 0,
    sca_table['rms_total'] / sca_table['alerts_2425'] * 100,
    np.nan
)

sca_table.drop(columns='geometry').sort_values(['in_coverage', 'rms_total'], ascending=[False, False]).head(20)



## Quality checks for the merged table

Three simple checks:
1. Make sure every SCA polygon is represented exactly once.
2. Confirm coverage is equivalent to `alerts_2425 > 0`.
3. Check that total RMS counts in the SCA table match totals from the filtered RMS dataset.


In [ ]:

assert sca_table['sca'].nunique() == len(sca_table), 'Duplicate SCA rows found.'
assert (sca_table['in_coverage'] == (sca_table['alerts_2425'] > 0)).all(), 'Coverage flag mismatch.'
assert sca_table['rms_total'].sum() == len(rms_shoot), 'RMS total mismatch after merge.'

print('All validation checks passed.')



## SCAs with no ShotSpotter alerts but substantial gun violence

These are the coverage-gap areas: no ShotSpotter alerts in 2024–2025, but at least 5 RMS shootings/homicides.


In [ ]:

coverage_gaps = (
    sca_table[(sca_table['alerts_2425'] == 0) & (sca_table['rms_total'] >= 5)]
    .drop(columns='geometry')
    .sort_values('rms_total', ascending=False)
)

coverage_gaps[['sca', 'Precinct', 'rms_2024', 'rms_2025', 'rms_total']]



## Map 1: RMS shootings/homicides by SCA

A choropleth of total RMS shootings/homicides in 2024–2025.


In [ ]:

fig, ax = plt.subplots(figsize=(9, 9))
sca_table.plot(
    column='rms_total',
    cmap='YlOrRd',
    linewidth=0.4,
    edgecolor='white',
    legend=True,
    legend_kwds={'label': 'RMS shootings/homicides (2024–2025)', 'shrink': 0.7},
    ax=ax,
    missing_kwds={'color': 'lightgrey'}
)
ax.set_title('RMS shootings/homicides by scout car area, 2024–2025')
ax.set_axis_off()
plt.tight_layout()
plt.show()



## Map 2: ShotSpotter coverage gaps

This map classifies each SCA into one of three groups:
- ShotSpotter coverage,
- no alerts and at least 5 RMS shootings/homicides,
- no alerts and fewer than 5 RMS shootings/homicides.


In [ ]:

sca_table['map_category'] = 'No alerts, <5 RMS'
sca_table.loc[sca_table['in_coverage'], 'map_category'] = 'ShotSpotter coverage'
sca_table.loc[(~sca_table['in_coverage']) & (sca_table['rms_total'] >= 5), 'map_category'] = 'No alerts, ≥5 RMS'

color_map = {
    'ShotSpotter coverage': '#01696f',
    'No alerts, ≥5 RMS': '#a13544',
    'No alerts, <5 RMS': '#d4d1ca'
}

fig, ax = plt.subplots(figsize=(9, 9))
for category, color in color_map.items():
    sca_table[sca_table['map_category'] == category].plot(
        color=color,
        linewidth=0.4,
        edgecolor='white',
        ax=ax,
        label=category
    )

handles = [mpatches.Patch(color=color, label=label) for label, color in color_map.items()]
ax.legend(handles=handles, loc='lower left', title='SCA status')
ax.set_title('ShotSpotter coverage gaps by scout car area')
ax.set_axis_off()
plt.tight_layout()
plt.show()



## Save outputs

Uncomment and run this cell if you want to write the merged SCA table to disk.


In [ ]:

# sca_table.drop(columns='geometry').to_csv('sca_analysis_2024_2025.csv', index=False)



## Add CVI overlap and compare four SCA groups

This section brings in the Community Violence Intervention (CVI) polygons and identifies which scout car areas intersect a CVI zone.

That allows the analysis to split SCAs into four groups:
- ShotSpotter and CVI
- ShotSpotter only
- CVI only
- Neither ShotSpotter nor CVI


In [ ]:

cvi_gdf = gpd.read_file('Resources/cvi_map.geojson')

# Reproject to match the SCA layer if needed
if cvi_gdf.crs != sca_gdf.crs:
    cvi_gdf = cvi_gdf.to_crs(sca_gdf.crs)

# Spatial join: flag SCAs that intersect any CVI polygon
sca_for_join = sca_gdf[['Area', 'Precinct', 'geometry']].rename(columns={'Area': 'sca'}).copy()
sca_for_join['sca'] = sca_for_join['sca'].astype(int)

sca_cvi_join = gpd.sjoin(
    sca_for_join[['sca', 'geometry']],
    cvi_gdf[['geometry']],
    how='left',
    predicate='intersects'
)

sca_cvi_join['in_cvi'] = sca_cvi_join['index_right'].notna()
sca_cvi = (
    sca_cvi_join.groupby('sca', as_index=False)
    .agg(in_cvi=('in_cvi', 'max'))
)

sca_cvi.head()



## Merge the CVI flag into the SCA table

This creates a combined SCA table with both intervention flags:
- `in_coverage` for ShotSpotter coverage
- `in_cvi` for overlap with a CVI polygon


In [ ]:

sca_table = sca_table.merge(sca_cvi, on='sca', how='left')
sca_table['in_cvi'] = sca_table['in_cvi'].fillna(False)

print('SCAs overlapping a CVI zone:', int(sca_table['in_cvi'].sum()))
sca_table[['sca', 'Precinct', 'in_coverage', 'in_cvi', 'rms_2024', 'rms_2025']].head()



## Create the four-group comparison

Each SCA is assigned to one of four mutually exclusive groups. Then the notebook compares 2024 and 2025 shooting/homicide totals across those groups.


In [ ]:

def assign_group(row):
    if row['in_coverage'] and row['in_cvi']:
        return 'ShotSpotter & CVI'
    if row['in_coverage'] and not row['in_cvi']:
        return 'ShotSpotter only'
    if (not row['in_coverage']) and row['in_cvi']:
        return 'CVI only'
    return 'No ShotSpotter / No CVI'

sca_table['group_4way'] = sca_table.apply(assign_group, axis=1)
sca_table['group_4way'].value_counts()


In [ ]:

# Convert the SCA-level wide table into a long format for year-by-year group summaries
rows = []
for _, row in sca_table.drop(columns='geometry').iterrows():
    rows.append({'sca': row['sca'], 'group_4way': row['group_4way'], 'year': 2024, 'shootings': row['rms_2024']})
    rows.append({'sca': row['sca'], 'group_4way': row['group_4way'], 'year': 2025, 'shootings': row['rms_2025']})

sca_group_long = pd.DataFrame(rows)

four_group_summary = (
    sca_group_long.groupby(['group_4way', 'year'])['shootings']
    .sum()
    .reset_index()
)

four_group_pivot = (
    four_group_summary.pivot(index='group_4way', columns='year', values='shootings')
    .fillna(0)
    .astype(int)
)

four_group_pivot['pct_change_2024_2025'] = (
    (four_group_pivot[2025] - four_group_pivot[2024]) /
    four_group_pivot[2024].replace(0, np.nan)
) * 100

four_group_pivot.sort_index()



## Visualize the four-group trend comparison

This chart makes it easier to see which combinations of ShotSpotter coverage and CVI overlap saw the biggest changes from 2024 to 2025.


In [ ]:

plot_order = [
    'ShotSpotter & CVI',
    'ShotSpotter only',
    'CVI only',
    'No ShotSpotter / No CVI'
]

plot_df = four_group_pivot.loc[plot_order, [2024, 2025]].copy()

fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(plot_order))
width = 0.35

bars1 = ax.bar(x - width/2, plot_df[2024], width, label='2024', color='#01696f')
bars2 = ax.bar(x + width/2, plot_df[2025], width, label='2025', color='#da7101')

for bars in [bars1, bars2]:
    for bar in bars:
        ax.text(
            bar.get_x() + bar.get_width()/2,
            bar.get_height() + 4,
            f'{int(bar.get_height())}',
            ha='center',
            va='bottom',
            fontsize=9
        )

ax.set_xticks(x)
ax.set_xticklabels(plot_order, rotation=15, ha='right')
ax.set_ylabel('RMS shootings / homicides')
ax.set_title('2024 vs. 2025 shootings/homicides by ShotSpotter/CVI group')
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
sca_table.plot(column='group_4way', legend=True, figsize=(9, 9), cmap='Set2', edgecolor='white')

**Note:** One year-over-year comparison is not super robust, but it's the cleanest data that we're able to get right now. The real test would be to do a before-and-after test (DiD, maybe) in each of those 4 groups of scout car areas.